In [ ]:
import numpy as np
import xarray as xr
from numpy.polynomial.legendre import Legendre
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
def ax_pos_inch_to_absolute(fig_size, ax_pos_inch):
    ax_pos_absolute = []
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])

    return ax_pos_absolute

In [ ]:
def ax_pos_cm_to_absolute(fig_size, ax_pos_cm):
    ax_pos_absolute = []
    ax_pos_inch = [ pos / 2.54 for pos in ax_pos_cm ]
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])
    
    return ax_pos_absolute

In [ ]:
def angle_to_l(x):
    """Vectorized 1/x, treating x==0 manually"""
    x = np.array(x, float)
    near_zero = np.isclose(x, 0)
    x[near_zero] = np.inf
    x[~near_zero] = 180. / x[~near_zero]
    return x


In [ ]:
def freq_to_time(x):
    """Vectorized 1/x, treating x==0 manually"""
    x = np.array(x, float)
    near_zero = np.isclose(x, 0)
    x[near_zero] = np.inf
    x[~near_zero] = 1. / x[~near_zero]
    return x


In [ ]:
# equation parameters
tau_0 = 2.3
epsilon_0 = 5.8
gamma_0 = 0.060

In [ ]:
# grid
ntrunc = 1000

yg, wg = np.polynomial.legendre.leggauss(deg=1000)

xg = np.linspace(np.pi, 0, 1001)
yg = np.cos(xg)

omega = np.logspace(np.log10(1./360), 0, 20) * 2 * np.pi

colatr = np.arccos(yg)
colatd = np.rad2deg(colatr)


In [ ]:
# auto-covariance
C = np.zeros((len(yg), len(omega)))

for j in range(len(omega)):
    for n in range(1, ntrunc):
        P = Legendre.basis(n)
        C[:, j] += epsilon_0**2 * (2 * n + 1) / (4 * np.pi) * P(yg) / ( omega[j]**2 * tau_0**2 + ( gamma_0**2 * n * (n+1) + 1 )**2 )


In [ ]:
# scale to get correlation
for j in range(len(omega)):
    C[:, j] /= C[-1, j]

In [ ]:
# auto-covariance of the forcing
C_S = np.zeros(len(yg))

for n in range(ntrunc):
    P = Legendre.basis(n)
    C_S += epsilon_0**2 * (2 * n + 1) / (4 * np.pi) * P(yg) * tau_0 / ( gamma_0**2 * n * (n+1) + 1 )


In [ ]:
# forcing covariance is independent of frequency
C_S_w = np.zeros((len(yg), len(omega)))

for j in range(len(omega)):
    C_S_w[:, j] = C_S[:] / C_S[-1]

In [ ]:
fig_size = (14.20/2.54, 17.50/2.54)
fig = plt.figure(figsize=fig_size)

ax = []

ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.75, 10.00, 12.00, 06.00])))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.75, 02.50, 12.00, 06.00])))


##### ax0 #####
ax[0].grid()

cs0 = ax[0].contourf(colatr / gamma_0, omega / 2 / np.pi, C.T, levels=np.arange(0.1, 1.1, 0.1), extend='neither', cmap='Greens')


ax[0].set_xlabel(r'$\theta a / \lambda_{0}$', fontsize=10)
ax[0].set_ylabel(r'Frequency ($\omega$) [cpd]', fontsize=10)
# ax[0].set_xscale('log')
ax[0].set_yscale('log')
ax[0].set_xlim(0, 3)
ax[0].set_ylim(1/360, 1)

ax[0].text(-00.05, 01.05, 'A', ha='right', va='bottom', transform=ax[0].transAxes, fontsize=10, fontweight='bold', color='black')
ax[0].text(00.95, 00.95, 'Response', ha='right', va='top', transform=ax[0].transAxes, fontsize=10, fontweight='bold', color='black')


##### ax1 #####
ax[1].grid()

cs1 = ax[1].contourf(colatr / gamma_0, omega / 2 / np.pi, C_S_w.T, levels=np.arange(0.1, 1.1, 0.1), extend='neither', cmap='Greens')

cax = fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.75, 00.75, 10.00, 00.25])) 
cbar = fig.colorbar(cs1, cax=cax, orientation='horizontal', extend='neither')
cax.locator_params(nbins=8)

ax[1].set_xlabel(r'$\theta a / \lambda_{0}$', fontsize=10)
ax[1].set_ylabel(r'Frequency ($\omega$) [cpd]', fontsize=10)
# ax[0].set_xscale('log')
ax[1].set_yscale('log')
ax[1].set_xlim(0, 3)
ax[1].set_ylim(1/360, 1)

ax[1].text(-00.05, 01.05, 'B', ha='right', va='bottom', transform=ax[1].transAxes, fontsize=10, fontweight='bold', color='black')
ax[1].text(00.95, 00.95, 'Forcing', ha='right', va='top', transform=ax[1].transAxes, fontsize=10, fontweight='bold', color='black')


In [ ]:
# base dir
base_dir = (Path.cwd() / "../../").resolve()
save_dir = base_dir / "figures"

In [ ]:
file_name = "fig-s03"
Path(save_dir).mkdir(parents=True, exist_ok=True)
fig.savefig(str(save_dir / file_name) + ".png", dpi=600, format='png', facecolor='white')
fig.savefig(str(save_dir / file_name) + ".pdf", dpi=600, format='pdf', facecolor='white')